In [1]:
import sys
sys.path.append('..')
from genoml.utils import *

In [2]:
from Bio import SeqIO

def fasta_to_df(fasta_path):
    records = [
        {"id": record.id, "seq": str(record.seq)}
        for record in SeqIO.parse(fasta_path, "fasta")
    ]
    return pd.DataFrame(records)

df = fasta_to_df("../data/TFBU_MPRA/raw/library1_final_edit50-0509.fasta")
df['seq'] = df['seq'].str.slice(15, -15)
df.to_csv('../data/TFBU_9cell/raw/lib.tsv', sep='\t', index=False)

df

,id,seq
0,0.7488720799999999_ARNT2_pos_00_chr17_43361125...,ACTGGCCGCTTGACGCGGATTTAAGCGTGTAAACGCAACACACAAC...
1,0.7244516230000001_ARNT2_pos_00_chr3_190424655...,ACTGGCCGCTTGACGTCGCCTCCCTCAGACTCTGCACCCGCCACAA...
2,0.697715514_ARNT2_pos_00_chr20_46174909_461750...,ACTGGCCGCTTGACGCGAAGTCGTAGAGGCTCCGCAGCGCCGACAT...
3,0.6746311209999999_ARNT2_pos_00_chr20_31968460...,ACTGGCCGCTTGACGTTTGGTCCGGAAGGCGCCGGCGCTGTCCTTG...
4,0.664452299_ARNT2_pos_00_chr22_41666645_416668...,ACTGGCCGCTTGACGCGCGATGGCCGCGGTAGAAGTCAAGCGCCAG...
...,...,...
17995,verified_sequence_51,ACTGGCCGCTTGACGCCGAAGTTGGCGCCTGCGGTGCGGCCGCCGC...
17996,verified_sequence_52,ACTGGCCGCTTGACGCGGCGCCGGCGCGGCGTGGTACCGTAAGCCG...
17997,verified_sequence_53,ACTGGCCGCTTGACGTACTTGAGCCCAGGAGTTGAGACCAGCCTAA...
17998,verified_sequence_54,ACTGGCCGCTTGACGCCCTACACACATACAGCCTCCCTCATTATCA...


In [8]:
cell_types = ['HEK293T', 'HeLa-S3', 'HepG2', 'HL-60', 'Jurkat', 'K562', 'MCF-7', 'PANC-1', 'Raji']
cell_names = ['293T', 'HelaS3', 'HepG2', 'HL60', 'JurkatN251119', 'K562', 'MCF7', 'PANC', 'Raji_N']

df = pd.read_csv('../data/TFBU_9cell/raw/lib.tsv', sep='\t')

dfs = [
    pd.read_csv(f'../data/TFBU_9cell/raw/7_exp_{cell_name}_mean.tsv', sep='\t', header=None, names=['seq', cell_name, 'id'])
    for cell_name in cell_names
]

for ct, df_ in zip(cell_types, dfs):
    print(ct, df_.shape)

for df_ in dfs:
    df = df.merge(df_, on=["seq", "id"], how="left")

rename_map = dict(zip(cell_names, cell_types))
df = df.rename(columns=rename_map)

df[cell_types] = np.log2(df[cell_types]+1e-6)
df.describe()

HEK293T (16942, 3)
HeLa-S3 (16795, 3)
HepG2 (16911, 3)
HL-60 (16778, 3)
Jurkat (16842, 3)
K562 (16782, 3)
MCF-7 (16951, 3)
PANC-1 (16949, 3)
Raji (16827, 3)


,HEK293T,HeLa-S3,HepG2,HL-60,Jurkat,K562,MCF-7,PANC-1,Raji
count,16942.000,16795.000,16911.000,16778.000,16842.000,16782.000,16951.000,16949.000,16827.000
mean,-1.168,-0.631,-0.730,-0.346,-1.037,-1.764,-0.009,-0.872,-0.872
std,1.262,0.775,1.156,0.681,1.383,1.868,0.488,0.932,1.317
min,-3.813,-2.904,-2.580,-2.581,-4.374,-6.029,-1.848,-3.058,-4.459
25%,-2.003,-1.106,-1.562,-0.696,-2.049,-3.163,-0.268,-1.529,-1.895
50%,-1.615,-0.843,-1.144,-0.510,-1.399,-2.222,-0.137,-1.128,-1.170
75%,-0.793,-0.406,-0.230,-0.255,-0.292,-0.700,0.069,-0.451,-0.064
max,8.135,9.766,5.756,4.982,4.781,5.265,7.016,9.230,5.413


In [9]:
df.to_csv('../data/TFBU_9cell/data.tsv', sep='\t', index=False)